# Task
Generate a trivia quiz with different categories of questions, including 'News of the Week', 'British Culture', 'French Culture', 'Theme Guessing', 'Alphabetical Answers', 'Ranking', and 'General Knowledge' questions. The generated questions should be unique and stored in a CSV file. Additionally, create a PowerPoint presentation containing only the questions and a Word document with both questions and answers, both organized by their respective sections and category order.

In [ ]:
!pip install python-pptx python-docx groq google-api-python-client
import pandas as pd
import datetime
import csv
import os
import re
from IPython.display import display, HTML
from pptx import Presentation
from pptx.util import Inches
from docx import Document
from docx.shared import Inches
from transformers import pipeline
import json
import random
from groq import Groq
from google.colab import userdata



print("All required libraries loaded successfully.")

All required libraries loaded successfully.


## Load Existing Quiz Data

### Subtask:
Load any existing quiz data from 'trivia_quiz.csv' into a pandas DataFrame, or initialize a new empty DataFrame if the file does not exist. This will be the base for adding new questions.


**Reasoning**:
Call the `load_existing_quiz_data` function again to load the now existing `trivia_quiz.csv` into `quiz_df` and display its head as instructed.



In [ ]:
import pandas as pd
import os

def load_existing_quiz_data():
    csv_file_name = 'trivia_quiz.csv'
    expected_columns = ["Category", "Question", "Answer", "Generated_Date", "Source"]

    if os.path.exists(csv_file_name):
        try:
            df = pd.read_csv(csv_file_name)
            # If the file was read but resulted in an empty DataFrame or incorrect columns,
            # re-initialize with the expected columns.
            if df.empty or not all(col in df.columns for col in expected_columns):
                print(f"'{csv_file_name}' exists but is empty or has incorrect columns. Initializing new empty DataFrame with specified columns.")
                df = pd.DataFrame(columns=expected_columns)
            else:
                print(f"Loading existing quiz data from {csv_file_name}")
        except pd.errors.EmptyDataError:
            print(f"'{csv_file_name}' exists but is empty. Initializing new empty DataFrame with specified columns.")
            df = pd.DataFrame(columns=expected_columns)
        except Exception as e:
            print(f"An error occurred while reading '{csv_file_name}': {e}. Initializing new empty DataFrame with specified columns.")
            df = pd.DataFrame(columns=expected_columns)
    else:
        print(f"'{csv_file_name}' not found. Initializing new empty DataFrame with specified columns.")
        df = pd.DataFrame(columns=expected_columns)
    return df

quiz_df = load_existing_quiz_data()
print("Loaded quiz DataFrame head after previous operations:")
print(quiz_df.head())

Loading existing quiz data from trivia_quiz.csv
Loaded quiz DataFrame head after previous operations:
          Category                                           Question  \
0  British Culture  Who is the British artist behind the sculpture...   
1  British Culture  What is the name of the famous British music f...   
2  British Culture  Which British author wrote the novel 'Brideshe...   
3  British Culture  Which prehistoric monument in England is belie...   
4  British Culture  In British culture, what is the traditional na...   

             Answer Generated_Date                  Source  
0    Antony Gormley     2026-05-24  AI generated with Groq  
1       Glastonbury     2026-05-24  AI generated with Groq  
2      Evelyn Waugh     2026-05-24  AI generated with Groq  
3        Stonehenge     2026-05-24  AI generated with Groq  
4  Christmas Dinner     2026-05-24  AI generated with Groq  


## Modify and Generate 'British Culture' Questions

### Subtask:
Generate 5 unique questions and corresponding answers for the 'British Culture' category, ensuring they are not duplicates of existing questions in the DataFrame.


In [ ]:
# Load key from Colab Secrets
GOOGLE_API_KEY = userdata.get('Pubquizkey')

# Initialize Groq client
client = Groq(api_key=GOOGLE_API_KEY)

# Define the Groq model to use
GROQ_MODEL = "llama-3.3-70b-versatile"

category = "British Culture"
source = "AI generated with Groq"

generated_date = datetime.date.today().strftime("%Y-%m-%d")

def generate_ai_questions(category, n_questions=5):
    prompt = f"""
    Generate {n_questions} different pub quiz questions about {category}.

    Rules:
    - Questions must be different every time.
    - Do not use overly obvious repeated questions.
    - Each question must have one clear answer.
    - Return only valid JSON.
    - Format:
    [
      {{
        "Question": "...",
        "Answer": "..."
      }}
    ]
    """

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    text = response.choices[0].message.content.strip()

    # Clean possible markdown, sometimes Groq might return markdown despite instructions
    text = text.replace("```json", "").replace("```", "").strip()

    return json.loads(text)


# Generate questions with AI
british_culture_questions = generate_ai_questions(
    category=category,
    n_questions=5
)

random.shuffle(british_culture_questions)

for q_data in british_culture_questions:
    question = q_data["Question"]
    answer = q_data["Answer"]

    # Check uniqueness
    is_unique = not (
        (quiz_df["Question"].str.lower() == question.lower()) &
        (quiz_df["Answer"].str.lower() == answer.lower())
    ).any()

    if is_unique:
        new_question = pd.DataFrame([{
            "Category": category,
            "Question": question,
            "Answer": answer,
            "Generated_Date": generated_date,
            "Source": source
        }])

        quiz_df = pd.concat([quiz_df, new_question], ignore_index=True)

        print(f"Added new '{category}' question:")
        print(f"Question: {question}\n")
        print(f"Answer: {answer}\n")

    else:
        print(f"Question already exists, skipping: {question}")


print("\nUpdated quiz DataFrame head:")
print(quiz_df.head(15))

Added new 'British Culture' question:
Question: Who is the British artist behind the painting 'The Fighting Temeraire', which hangs in the National Gallery in London?

Answer: J.M.W. Turner

Added new 'British Culture' question:
Question: Which British band's 1975 album 'A Night at the Opera' is often cited as one of the greatest albums of all time?

Answer: Queen

Question already exists, skipping: Which British author wrote the dystopian novel 'A Clockwork Orange'?
Added new 'British Culture' question:
Question: In British culture, what is the traditional name for the day after Christmas Day?

Answer: Boxing Day

Added new 'British Culture' question:
Question: What is the name of the famous British prehistoric monument located in Wiltshire, England?

Answer: Stonehenge


Updated quiz DataFrame head:
           Category                                           Question  \
0   British Culture  Who is the British artist behind the sculpture...   
1   British Culture  What is the name o

**Reasoning**:
I need to define 5 unique British Culture questions and their answers, then loop through them, checking for uniqueness against the `quiz_df`, and add them if they are new. This will fulfill the subtask's requirement.



## Modify and Generate 'French Culture' Questions

### Subtask:
Generate 5 unique questions and corresponding answers for the 'French Culture' category, ensuring they are not duplicates of existing questions in the DataFrame.


In [ ]:
category = "French Culture"
source = "AI generated with Groq"

generated_date = datetime.date.today().strftime("%Y-%m-%d")

def generate_ai_questions(category, n_questions=5):
    prompt = f"""
    Generate {n_questions} different pub quiz questions about {category}.

    Rules:
    - Questions must be different every time.
    - Do not use overly obvious repeated questions.
    - Each question must have one clear answer.
    - Return only valid JSON.
    - Format:
    [
      {{
        "Question": "...",
        "Answer": "..."
      }}
    ]
    """

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    text = response.choices[0].message.content.strip()

    # Clean possible markdown, sometimes Groq might return markdown despite instructions
    text = text.replace("```json", "").replace("```", "").strip()

    return json.loads(text)


# Generate questions with AI
french_culture_questions = generate_ai_questions(
    category=category,
    n_questions=5
)

random.shuffle(french_culture_questions)

for q_data in french_culture_questions:
    question = q_data["Question"]
    answer = q_data["Answer"]

    # Check uniqueness
    is_unique = not (
        (quiz_df["Question"].str.lower() == question.lower()) &
        (quiz_df["Answer"].str.lower() == answer.lower())
    ).any()

    if is_unique:
        new_question = pd.DataFrame([{
            "Category": category,
            "Question": question,
            "Answer": answer,
            "Generated_Date": generated_date,
            "Source": source
        }])

        quiz_df = pd.concat([quiz_df, new_question], ignore_index=True)

        print(f"Added new '{category}' question:")
        print(f"Question: {question}")
        print(f"Answer: {answer}\n")

    else:
        print(f"Question already exists, skipping: {question}")


print("\nUpdated quiz DataFrame head:")
print(quiz_df.head(20))

Added new 'French Culture' question:
Question: What is the name of the traditional French culinary term for a high-quality, three-Michelin-starred restaurant?
Answer: Haute Cuisine

Added new 'French Culture' question:
Question: What is the name of the famous French literary movement that emerged in the 19th century, characterized by its emphasis on emotion and imagination?
Answer: Romanticism

Added new 'French Culture' question:
Question: Who is the French author of the classic novel 'Les Misérables'?
Answer: Victor Hugo

Added new 'French Culture' question:
Question: Which French artist is famous for his painting 'The Gleaners'?
Answer: Jean-François Millet

Added new 'French Culture' question:
Question: Which French city is known as the 'Capital of Light' and is home to the famous Festival of Lights?
Answer: Lyon


Updated quiz DataFrame head:
                Category                                           Question  \
0        British Culture  Who is the British artist behind th

## Generate 'News of 2026' Questions

### Subtask:
Generate 5 unique questions and corresponding answers for the 'News of the Week' category using the AI function, ensuring they are not duplicates of existing questions in the DataFrame.

In [ ]:
category = "News of 2026"
source = "AI generated with Groq"

generated_date = datetime.date.today().strftime("%Y-%m-%d")

# Generate questions with AI
news_of_week_questions = generate_ai_questions(
    category=category,
    n_questions=5
)

random.shuffle(news_of_week_questions)

for q_data in news_of_week_questions:
    question = q_data["Question"]
    answer = q_data["Answer"]

    # Check uniqueness
    is_unique = not (
        (quiz_df["Question"].str.lower() == question.lower()) &
        (quiz_df["Answer"].str.lower() == answer.lower())
    ).any()

    if is_unique:
        new_question = pd.DataFrame([{
            "Category": category,
            "Question": question,
            "Answer": answer,
            "Generated_Date": generated_date,
            "Source": source
        }])

        quiz_df = pd.concat([quiz_df, new_question], ignore_index=True)

        print(f"Added new '{category}' question:")
        print(f"Question: {question}")
        print(f"Answer: {answer}\n")

    else:
        print(f"Question already exists, skipping: {question}")


print("\nUpdated quiz DataFrame head:")
print(quiz_df.head(25))

Added new 'News of 2026' question:
Question: Which social media platform did the European Union fine in 2026 for violating the General Data Protection Regulation?
Answer: Facebook

Added new 'News of 2026' question:
Question: Who was elected as the new leader of the Conservative Party in the United Kingdom in 2026?
Answer: Rishi Sunak

Added new 'News of 2026' question:
Question: What significant astronomical event occurred on April 8, 2026, that was visible from certain parts of the world?
Answer: A total solar eclipse

Added new 'News of 2026' question:
Question: What was the name of the spacecraft that successfully landed on Mars in 2026 as part of a joint mission between NASA and the European Space Agency?
Answer: ExoMars

Added new 'News of 2026' question:
Question: In 2026, what was the name of the hurricane that made landfall in the United States, causing widespread damage and flooding?
Answer: Hurricane Idalia


Updated quiz DataFrame head:
                Category             

## Modify and Generate 'Theme Guessing' Questions

### Subtask:
Ensure 4 unique questions and answers are generated for the 'Theme Guessing' category, checking for uniqueness against existing questions.


**Reasoning**:
I need to explicitly define 4 unique 'Theme Guessing' questions and answers, then iterate through them to add them to the `quiz_df` after checking for uniqueness, and finally display the updated DataFrame.



In [ ]:
category = "Theme Guessing"
source = "AI generated with Groq"

generated_date = datetime.date.today().strftime("%Y-%m-%d")

# Generate questions with AI
theme_guessing_questions = generate_ai_questions(
    category=category,
    n_questions=4
)

random.shuffle(theme_guessing_questions)

for q_data in theme_guessing_questions:
    question = q_data["Question"]
    answer = q_data["Answer"]

    # Check uniqueness
    is_unique = not (
        (quiz_df["Question"].str.lower() == question.lower()) &
        (quiz_df["Answer"].str.lower() == answer.lower())
    ).any()

    if is_unique:
        new_question = pd.DataFrame([{
            "Category": category,
            "Question": question,
            "Answer": answer,
            "Generated_Date": generated_date,
            "Source": source
        }])

        quiz_df = pd.concat([quiz_df, new_question], ignore_index=True)

        print(f"Added new '{category}' question:")
        print(f"Question: {question}")
        print(f"Answer: {answer}\n")

    else:
        print(f"Question already exists, skipping: {question}")


print("\nUpdated quiz DataFrame head:")
print(quiz_df.head(30))

Added new 'Theme Guessing' question:
Question: Which British rock band, formed in 1960, is known for hits like 'I Want to Hold Your Hand', 'Yesterday', and 'Hey Jude'?
Answer: The Beatles

Added new 'Theme Guessing' question:
Question: Which Dutch post-impressionist painter is famous for works like 'Sunflowers', 'The Starry Night', and 'Self-Portrait with Bandaged Ear'?
Answer: Vincent van Gogh

Added new 'Theme Guessing' question:
Question: In Greek mythology, who was the god of the underworld, often depicted with a helmet of darkness and a key to the gates of the underworld?
Answer: Hades

Added new 'Theme Guessing' question:
Question: Which American author wrote the novels 'The Shining', 'Misery', and 'Carrie', all of which have been adapted into successful films?
Answer: Stephen King


Updated quiz DataFrame head:
                Category                                           Question  \
0        British Culture  Who is the British artist behind the sculpture...   
1        Bri

## Modify and Generate 'Alphabetical Answers' Questions

### Subtask:
Update the code to generate 6 unique questions and answers for the 'Alphabetical Answers' category, ensuring they are not duplicates of existing questions in the DataFrame.


**Reasoning**:
I need to define 6 unique 'Alphabetical Answers' questions and their answers, then loop through them, checking for uniqueness against the `quiz_df`, and add them if they are new. This will fulfill the subtask's requirement.



In [ ]:
category = "Alphabetical Answers"
source = "AI generated with Groq"

generated_date = datetime.date.today().strftime("%Y-%m-%d")

# Generate questions with AI
alphabetical_questions = generate_ai_questions(
    category=category,
    n_questions=6
)

random.shuffle(alphabetical_questions)

for q_data in alphabetical_questions:
    question = q_data["Question"]
    answer = q_data["Answer"]

    # Check uniqueness
    is_unique = not (
        (quiz_df["Question"].str.lower() == question.lower()) &
        (quiz_df["Answer"].str.lower() == answer.lower())
    ).any()

    if is_unique:
        new_question = pd.DataFrame([{
            "Category": category,
            "Question": question,
            "Answer": answer,
            "Generated_Date": generated_date,
            "Source": source
        }])

        quiz_df = pd.concat([quiz_df, new_question], ignore_index=True)

        print(f"Added new '{category}' question:")
        print(f"Question: {question}")
        print(f"Answer: {answer}\n")

    else:
        print(f"Question already exists, skipping: {question}")


print("\nUpdated quiz DataFrame head:")
print(quiz_df.head(40))

Added new 'Alphabetical Answers' question:
Question: Which chemical element has the symbol 'Xe'?
Answer: Xenon

Added new 'Alphabetical Answers' question:
Question: Which planet in our solar system starts with the letter 'V'?
Answer: Venus

Added new 'Alphabetical Answers' question:
Question: Which artist painted 'The Yellow Christ'?
Answer: Yves Tanguy is not correct, the correct answer is 'No artist starts with the letter Y that painted a famous work', however one answer is 'Yosl Bergner did paint but an example here is', a correct answer for the quiz is 'Yves is incorrect, but an example could be  - 'Y' could be for the more unknown artist, 'Yue Minjun', yet the answer here to the 'The Yellow Christ' is actually 'Y' is not correct for this example - this was actually painted by 'Paul Gauguin', but to continue - an artist that starts with 'Y' that has painted is indeed 'Yayoi Kusama'

Added new 'Alphabetical Answers' question:
Question: Which Dutch painter is known for his works such

## Modify and Generate 'Ranking' Questions

### Subtask:
Update the code to generate 2 unique questions and answers for the 'Ranking' category, ensuring they are not duplicates of existing questions in the DataFrame.


**Reasoning**:
I will define two new ranking questions and add them to the `quiz_df` after checking for uniqueness, then display the updated DataFrame.



In [ ]:
category = "Ranking"
source = "AI generated with Groq"

generated_date = datetime.date.today().strftime("%Y-%m-%d")

# Generate questions with AI
ranking_questions = generate_ai_questions(
    category=category,
    n_questions=2
)

random.shuffle(ranking_questions)

for q_data in ranking_questions:
    question = q_data["Question"]
    answer = q_data["Answer"]

    # Check uniqueness
    is_unique = not (
        (quiz_df["Question"].str.lower() == question.lower()) &
        (quiz_df["Answer"].str.lower() == answer.lower())
    ).any()

    if is_unique:
        new_question = pd.DataFrame([{
            "Category": category,
            "Question": question,
            "Answer": answer,
            "Generated_Date": generated_date,
            "Source": source
        }])

        quiz_df = pd.concat([quiz_df, new_question], ignore_index=True)

        print(f"Added new '{category}' question:")
        print(f"Question: {question}")
        print(f"Answer: {answer}\n")

    else:
        print(f"Question already exists, skipping: {question}")


print("\nUpdated quiz DataFrame head:")
print(quiz_df.head(45))

Added new 'Ranking' question:
Question: Which ranking system is used to evaluate the quality of universities worldwide, considering factors such as research output, academic reputation, and international diversity?
Answer: QS World University Rankings

Added new 'Ranking' question:
Question: In the context of search engines, what is the term for the process of assigning a rank or position to a website in the search results based on its relevance and importance?
Answer: PageRank


Updated quiz DataFrame head:
                Category                                           Question  \
0        British Culture  Who is the British artist behind the sculpture...   
1        British Culture  What is the name of the famous British music f...   
2        British Culture  Which British author wrote the novel 'Brideshe...   
3        British Culture  Which prehistoric monument in England is belie...   
4        British Culture  In British culture, what is the traditional na...   
5         Fr

## Rename and Generate 'General Knowledge' Questions

### Subtask:
Rename the 'Random' category to 'General Knowledge' and generate 5 unique questions and answers for this new category, ensuring they are not duplicates of existing questions.


**Reasoning**:
This step renames the 'Random' category, defines a list of new 'General Knowledge' questions, iterates through them to check for uniqueness, and adds them to the DataFrame before displaying the updated head.



In [ ]:
category = "General Knowledge"
source = "AI generated with Groq"

generated_date = datetime.date.today().strftime("%Y-%m-%d")

# Generate questions with AI
general_knowledge_questions = generate_ai_questions(
    category=category,
    n_questions=5
)

random.shuffle(general_knowledge_questions)

for q_data in general_knowledge_questions:
    question = q_data["Question"]
    answer = q_data["Answer"]

    # Check uniqueness
    is_unique = not (
        (quiz_df["Question"].str.lower() == question.lower()) &
        (quiz_df["Answer"].str.lower() == answer.lower())
    ).any()

    if is_unique:
        new_question = pd.DataFrame([{
            "Category": category,
            "Question": question,
            "Answer": answer,
            "Generated_Date": generated_date,
            "Source": source
        }])

        quiz_df = pd.concat([quiz_df, new_question], ignore_index=True)

        print(f"Added new '{category}' question:")
        print(f"Question: {question}")
        print(f"Answer: {answer}\n")

    else:
        print(f"Question already exists, skipping: {question}")


print("\nUpdated quiz DataFrame head:")
print(quiz_df.head(50))

Added new 'General Knowledge' question:
Question: Who is the author of the famous novel 'To Kill a Mockingbird'?
Answer: Harper Lee

Added new 'General Knowledge' question:
Question: What is the largest living structure on Earth?
Answer: The Great Barrier Reef

Question already exists, skipping: Which planet in our solar system has the most moons?
Question already exists, skipping: What is the chemical symbol for gold?
Added new 'General Knowledge' question:
Question: Which artist painted the famous painting 'The Starry Night'?
Answer: Vincent van Gogh


Updated quiz DataFrame head:
                Category                                           Question  \
0        British Culture  Who is the British artist behind the sculpture...   
1        British Culture  What is the name of the famous British music f...   
2        British Culture  Which British author wrote the novel 'Brideshe...   
3        British Culture  Which prehistoric monument in England is belie...   
4        Britis

## Verify Duplicates

### Subtask:
Check `quiz_df` for any duplicate questions.

In [ ]:
def check_for_duplicates(df):
    """
    Checks a DataFrame for duplicate questions (case-insensitive) and removes them.
    Notes: This function removes all but the first occurrence of duplicate questions.
    Generating truly *new* unique questions to replace the removed ones dynamically
    would require an active LLM or a more complex generation mechanism, which is not
    part of this function's scope given the current mock `generate_question_answer`.
    """
    initial_row_count = len(df)

    # Create a temporary DataFrame with lowercased 'Question' and 'Answer' for comparison
    temp_df = df.copy()
    temp_df['Question_lower'] = temp_df['Question'].str.lower()
    temp_df['Answer_lower'] = temp_df['Answer'].str.lower()

    # Identify duplicate rows (all occurrences)
    duplicates_mask = temp_df.duplicated(subset=['Question_lower', 'Answer_lower'], keep=False)

    if duplicates_mask.any():
        print("\n--- Duplicate Questions Found ---")
        # Display all duplicate occurrences for user's information
        display(temp_df[duplicates_mask].sort_values(by=['Question_lower', 'Answer_lower']))
        print("-----------------------------------")

        # Remove duplicates, keeping the first occurrence
        deduplicated_df = temp_df.drop_duplicates(subset=['Question_lower', 'Answer_lower'], keep='first')

        removed_count = initial_row_count - len(deduplicated_df)
        print(f"\n{removed_count} duplicate question(s) removed.")
        print("Note: To genuinely 'replace' removed duplicates with new, unique questions,")
        print("a sophisticated question generation mechanism (e.g., a live LLM API)")
        print("would be needed. The current `generate_question_answer` is a mock")
        print("and cycles through a fixed set of questions.")

        # Drop the temporary lowercased columns before returning
        deduplicated_df = deduplicated_df.drop(columns=['Question_lower', 'Answer_lower'])
        return deduplicated_df
    else:
        print("No duplicate questions found in the DataFrame.")
        # Drop the temporary lowercased columns before returning
        deduplicated_df = temp_df.drop(columns=['Question_lower', 'Answer_lower'])
        return deduplicated_df

# Check for duplicates in quiz_df and reassign the deduplicated result
quiz_df = check_for_duplicates(quiz_df)

# Explicitly compare quiz_df and previous_quiz_df after deduplication
if 'previous_quiz_df' in globals() and not previous_quiz_df.empty:
    print("\n--- Comparison with previous_quiz_df ---")
    current_questions = set(quiz_df.apply(lambda x: (x['Question'].lower(), x['Answer'].lower()), axis=1))
    previous_questions = set(previous_quiz_df.apply(lambda x: (x['Question'].lower(), x['Answer'].lower()), axis=1))

    new_unique_questions_count = len(current_questions - previous_questions)
    removed_from_previous_count = len(previous_questions - current_questions)

    if new_unique_questions_count > 0:
        print(f"Number of genuinely new unique questions added to quiz_df: {new_unique_questions_count}")
    else:
        print("No genuinely new unique questions were added that were not already present in previous_quiz_df.")

    if removed_from_previous_count > 0:
        print(f"Number of questions from previous_quiz_df no longer in current quiz_df: {removed_from_previous_count}")
        print("Note: This might occur if questions were modified or explicitly removed from the dataset during this session.")
    else:
        print("All unique questions from previous_quiz_df are still present in the current quiz_df.")

    print(f"Total unique questions in previous_quiz_df: {len(previous_questions)}")
    print(f"Total unique questions in current quiz_df: {len(current_questions)}")
    print("----------------------------------------")

No duplicate questions found in the DataFrame.


## Store All Questions in CSV

### Subtask:
Save the complete and updated quiz DataFrame, including all newly generated questions, to 'trivia_quiz.csv'. This will ensure persistence of all questions and their metadata.


**Reasoning**:
Save the current `quiz_df` to a CSV file, then load it back and display its head to confirm the save.



In [ ]:
import pandas as pd
import os

# Redefine load_existing_quiz_data to ensure it's available in case of kernel restart
def load_existing_quiz_data():
    csv_file_name_local = 'trivia_quiz.csv'
    expected_columns = ["Category", "Question", "Answer", "Generated_Date", "Source"]

    if os.path.exists(csv_file_name_local):
        try:
            df = pd.read_csv(csv_file_name_local)
            if df.empty or not all(col in df.columns for col in expected_columns):
                df = pd.DataFrame(columns=expected_columns)
        except pd.errors.EmptyDataError:
            df = pd.DataFrame(columns=expected_columns)
        except Exception:
            df = pd.DataFrame(columns=expected_columns)
    else:
        df = pd.DataFrame(columns=expected_columns)
    return df

# Ensure quiz_df is defined. If previous cells were not run, this will
# initialize an empty DataFrame or load from an existing CSV.
if 'quiz_df' not in globals():
    quiz_df = load_existing_quiz_data()
    print("Warning: 'quiz_df' was not found or was undefined. It has been re-initialized by loading 'trivia_quiz.csv' (or as an empty DataFrame if the CSV was not found/empty). To ensure all generated questions are included, please re-run the question generation cells (e.g., British Culture, French Culture, etc.) before saving.")

csv_file_name = 'trivia_quiz.csv'

# Save the DataFrame to CSV
quiz_df.to_csv(csv_file_name, index=False)

print(f"Questions saved to {csv_file_name}")

# Verify by loading the CSV back and displaying its head
loaded_df = pd.read_csv(csv_file_name)
print("\nVerifying saved CSV content (first 5 rows):")
print(loaded_df.head())

Questions saved to trivia_quiz.csv

Verifying saved CSV content (first 5 rows):
          Category                                           Question  \
0  British Culture  Who is the British artist behind the sculpture...   
1  British Culture  What is the name of the famous British music f...   
2  British Culture  Which British author wrote the novel 'Brideshe...   
3  British Culture  Which prehistoric monument in England is belie...   
4  British Culture  In British culture, what is the traditional na...   

             Answer Generated_Date                  Source  
0    Antony Gormley     2026-05-24  AI generated with Groq  
1       Glastonbury     2026-05-24  AI generated with Groq  
2      Evelyn Waugh     2026-05-24  AI generated with Groq  
3        Stonehenge     2026-05-24  AI generated with Groq  
4  Christmas Dinner     2026-05-24  AI generated with Groq  


## Reorder and Create PowerPoint Presentation and Word Document

### Subtask:
Generate a PowerPoint presentation ('trivia_quiz_questions.pptx') containing only the questions. The slides will be ordered by the categories: General Knowledge, News of the Week, British Culture, French Culture, Theme Guessing, Alphabetical Answers, Ranking, with each question on its own slide.


**Reasoning**:
I need to initialize a Presentation object, define the order of categories, then iterate through each category to create title slides and content slides with only the questions, and finally save the presentation.



In [ ]:
from pptx import Presentation
from pptx.util import Inches

# Define the desired order of categories
category_order = [
    'General Knowledge',
    'News of the Week',
    'British Culture',
    'French Culture',
    'Theme Guessing',
    'Alphabetical Answers',
    'Ranking'
]

prs = Presentation()

# Group questions by category for easier access
questions_by_category = quiz_df.groupby('Category')

for category in category_order:
    if category in questions_by_category.groups: # Check if the category exists in our data
        group = questions_by_category.get_group(category)

        # Add a title slide for each category
        slide_layout = prs.slide_layouts[0] # Title slide layout
        slide = prs.slides.add_slide(slide_layout)
        title = slide.shapes.title
        title.text = category

        # Add a content slide for questions for the current category
        slide_layout = prs.slide_layouts[1] # Title and content layout
        slide = prs.slides.add_slide(slide_layout)
        title = slide.shapes.title
        title.text = f"{category} Questions"

        body = slide.shapes.placeholders[1]
        tf = body.text_frame
        tf.clear() # Clear existing text in placeholder

        # Add questions to the slide
        for index, row in group.iterrows():
            p = tf.add_paragraph()
            p.text = f"- {row['Question']}"

# Save the presentation
pptx_file_name = 'trivia_quiz_questions.pptx'
prs.save(pptx_file_name)

print(f"PowerPoint presentation saved as {pptx_file_name}")

PowerPoint presentation saved as trivia_quiz_questions.pptx


In [ ]:
from docx import Document
from docx.shared import Inches

document = Document()

# Define the desired order of categories (already defined in previous cell, re-using for clarity)
category_order = [
    'General Knowledge',
    'News of the Week',
    'British Culture',
    'French Culture',
    'Theme Guessing',
    'Alphabetical Answers',
    'Ranking'
]

# Group questions by category for easier access
questions_by_category = quiz_df.groupby('Category')

for category in category_order:
    if category in questions_by_category.groups: # Check if the category exists in our data
        group = questions_by_category.get_group(category)

        # Add a heading for each category
        document.add_heading(category, level=1)

        # Add questions and answers for the current category
        for index, row in group.iterrows():
            document.add_paragraph(f"Question: {row['Question']}", style='List Bullet')
            document.add_paragraph(f"Answer: {row['Answer']}")
            document.add_paragraph('') # Add an empty line for spacing

# Save the Word document
docx_file_name = 'trivia_quiz_questions_with_answers.docx'
document.save(docx_file_name)

print(f"Word document saved as {docx_file_name}")

Word document saved as trivia_quiz_questions_with_answers.docx
